### Ensure necessary tools are present and working

In [15]:
import sklearn
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from econml.dml import CausalForestDML
from econml.validate import DRTester
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import shap

print(sklearn.__version__)

1.6.1


### Imports from existing scripts

In [2]:
import sys
from pathlib import Path
import os
sys.path.append(str(Path(os.getcwd()).parent))
Path(os.getcwd())

from scripts.pipeline.predictions.create_train_test_split import create_train_test_split
from scripts.shared.utils import load_trd_set, load_feature_matrix

### Creation of data set with one-hot categorical encodings

In [3]:
train_ids, test_ids = create_train_test_split()
trd_set = load_trd_set()
X_train, X_test = load_feature_matrix(train_ids), load_feature_matrix(test_ids)
# Don't iterate over the raw sets of train and test ids for y; use X_train and X_test index references to preserve order
y_train, y_test = np.array([1 if id in trd_set else 0 for id in X_train.index]), np.array([1 if id in trd_set else 0 for id in X_test.index])

print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)

# Drop all of the columns that are treatments whose effect we want to estimate
treatment_cols = ['polypharmacy_count', 'trials_BUPROPION', 'trials_MIRTAZAPINE', 'trials_SNRI', 'trials_SSRI', 'trials_VORTIOXETINE', 'augmentation_occured']
X_reduced_train = X_train.drop(columns=treatment_cols)
X_reduced_test = X_test.drop(columns=treatment_cols)
print(X_reduced_train.shape, X_reduced_test.shape)

# Concatenate training and testing into the entire population
X_full = pd.concat([X_train, X_test])
print(X_full.shape)

Found 9724 total patients.
Split into train set of size 7779 and test set of size 1945.
Train set TRD count: 827
Test set TRD count: 207
(7779, 59) (1945, 59) (7779,) (1945,)
(7779, 52) (1945, 52)
(9724, 59)


In [11]:
POLYPHARMACY_THRESHOLD=2
TRIALS_COLS=['trials_BUPROPION', 'trials_MIRTAZAPINE', 'trials_SNRI', 'trials_SSRI', 'trials_VORTIOXETINE']
SURVIVING_TREATMENTS=['polypharmacy', 'adequate_trial']
SURVIVING_TREATMENT_NAMES = {
    'polypharmacy': f'Medications active at index at least {POLYPHARMACY_THRESHOLD}',
    'adequate_trial': 'At least one medication arm having an adequate set of trials'
}

def get_treatment_indicator(patient_id: str, treatment: str) -> int:
    """Whether treatment was used on patient or not

    Args:
        patient_id (str): Respective patient
        treatment (str): Respective treatment

    Returns:
        int: Flag to indicate if treatment was applied to patient
    """
    patient_row = X_full.loc[patient_id]
    if treatment == 'polypharmacy':
        return int(patient_row['polypharmacy_count'] >= POLYPHARMACY_THRESHOLD)
    elif treatment == 'adequate_trial': # At least one arm of medication treatments had an adequate set of trials
        return int(patient_row[TRIALS_COLS].sum() >= 1)
    elif treatment == 'augmentation':
        return int(patient_row['augmentation_occured'])
    else:
        raise ValueError(f"Unrecognized treatment: {treatment}")

# For each treatment, determine if each train or test patient received it or not
T_train = {
    treatment: np.array([get_treatment_indicator(pid, treatment) for pid in X_reduced_train.index])
    for treatment in SURVIVING_TREATMENTS
}
T_test = {
    treatment: np.array([get_treatment_indicator(pid, treatment) for pid in X_reduced_test.index])
    for treatment in SURVIVING_TREATMENTS
}

for arr in T_train.values():
    print(arr.shape)

for arr in T_test.values():
    print(arr.shape)

(7779,)
(7779,)
(1945,)
(1945,)


In [5]:
cat_cols = ["Sex", "PreferredLanguage", "MaritalStatus", "Religion", "SmokingStatus", "Race_Ethnicity", "mdd_recurrence", "mdd_severity"]
X_reduced_full = pd.concat([X_reduced_train, X_reduced_test])
# Convert all categorical variables to one-hot encodings
X_encoded_full = pd.get_dummies(X_reduced_full, columns=cat_cols)
X_encoded_train, X_encoded_test = X_encoded_full.loc[X_reduced_train.index], X_encoded_full.loc[X_reduced_test.index]
print(X_encoded_train.shape, X_encoded_test.shape)

(7779, 81) (1945, 81)


### Implementation of causal random forest to estimate Conditional Average Treatment Effect (CATE)

In [9]:
causal_models, cate_test = {}, {}
seed = int(os.environ['SEED'])
save_folder = Path("figures")

for treatment in SURVIVING_TREATMENTS:
    model_y = RandomForestRegressor(random_state=seed, n_jobs=-1)
    model_t = RandomForestClassifier(random_state=seed, n_jobs=-1)
    n_estimators = 1000
    causal_forest = CausalForestDML(n_estimators=1000, model_y=model_y, model_t=model_t, random_state=seed, n_jobs=-1, discrete_treatment=True)
    # Fit with the train output, and whether this treatment was used on each patient in the training set
    causal_forest.fit(y_train, T_train[treatment], X=X_encoded_train, W=X_encoded_train)
    cate_test[treatment] = causal_forest.effect(X_encoded_test)
    causal_models[treatment] = causal_forest

os.makedirs(save_folder, exist_ok=True)
pd.DataFrame(cate_test, index=X_encoded_test.index).to_parquet(save_folder / "cate_test.parquet")

In [13]:
# For each treatment, create a histogram describing its importance over all patients
for treatment in SURVIVING_TREATMENTS:
    effect_vector = cate_test[treatment]
    fig, ax = plt.subplots()
    ax.hist(effect_vector, bins=100)
    ax.axvline(effect_vector.mean(), color='red', linestyle="--", label='Average effect')
    ax.axvline(0, color='green', linestyle='--', label='No effect')
    ax.set_xlabel("Estimated CATE on P(TRD)")
    ax.set_ylabel("Number of patients")
    ax.set_title(SURVIVING_TREATMENT_NAMES[treatment])
    ax.legend()
    fig.savefig(save_folder / f"{treatment}_CATE_histogram.png")
    plt.close(fig)

### Perform Heterogeneity tests

In [19]:
dr_testers = {}
cal_results = {}

for treatment in SURVIVING_TREATMENTS:
    tester = DRTester(
        model_regression=RandomForestClassifier(random_state=seed, n_jobs=-1),
        model_propensity=RandomForestClassifier(random_state=seed, n_jobs=-1),
        cate=causal_models[treatment],
        cv=5,
    )
    tester.fit_nuisance(
        Xval=X_encoded_test.to_numpy(),
        Dval=T_test[treatment],
        yval=y_test,
        Xtrain=X_encoded_train.to_numpy(),
        Dtrain=T_train[treatment],
        ytrain=y_train
    )
    dr_testers[treatment] = tester
    cal_results[treatment] = tester.evaluate_cal(Xval=X_encoded_test, Xtrain=X_encoded_train, n_groups=10)
    print(f"Treatment {SURVIVING_TREATMENT_NAMES[treatment]}:\n{cal_results[treatment].summary()}")

Treatment Medications active at index at least 2:
   treatment  cal_r_squared
0          1          0.045
Treatment At least one medication arm having an adequate set of trials:
   treatment  cal_r_squared
0          1         -0.126
